In [2]:
!nvidia-smi

Thu Apr 16 13:25:54 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   50C    P8             13W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
from google.colab import drive
drive.mount('/content/drive')
!ls

Mounted at /content/drive
drive  sample_data


In [4]:
%cd ./drive/MyDrive/ColabNotebooks

/content/drive/MyDrive/ColabNotebooks


In [5]:
!ls

Bologna-512.pgm
convert_to_ppm.py
DFTcudanaive.cu
DFTcudaOpt1.cu
DFTcudaOpt2.cu
DFTcudaOpt3.cu
DFTcudaOpt4.cu
DFTcudaOpt5.cu
dftnaive
dftopt1
dftopt2
dftopt3
dftopt4
dftopt5
edge_detection
edge_detection.h
edge_detection_kernel.cu
edge_detection_kernel.o
hello
hello_world.cu
image_to_pgm.cu
img_to_pgm
lagopontini.ppm
main.cu
main.o
NSB.jpg
NSB.pgm
NSB.ppm
nsight-compute-2025.2.1_2025.2.1.3-1_amd64.deb
NsightSystems-linux-cli-public-2026.1.1.204-3717666.deb
output_cuda-512.pgm
output_gray.jpg
result.ppm
rgb2gray
rgb2gray.cu
stb_image.h
stb_image_write.h
test_report_opt1.ncu-rep
test_report_opt2.ncu-rep
test_report_opt3.ncu-rep
test_report_opt4.ncu-rep
test_report_opt5.ncu-rep
Untitled0.ipynb


# Esecuzione conversione da formato qualsiasi a ppm
### eseguire in caso si voglia cambiare immagine da testare

In [5]:
!python3 convert_to_ppm.py NSB.jpg NSB.ppm

Immagine originale: JPEG, Mode: RGB, Size: (970, 546)
Dopo conversione: Mode: RGB, Size: (970, 546)
✓ Convertito con successo in NSB.ppm
Ora esegui: ./image_to_pgm NSB.ppm output.pgm


## compila solo in caso di:
### modifiche al codice per conversione da ppm a pgm
### testing su immagine differente

In [6]:
!nvcc -O3 -lineinfo image_to_pgm.cu -o img_to_pgm

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
image_to_pgm.cu(59): warning #1650-D: result of call is not used
      fscanf(file, "%2s", magic);
      ^

Remark: The warnings can be suppressed with "-diag-suppress <warning-number>"

image_to_pgm.cu(79): warning #1650-D: result of call is not used
      fscanf(file, "%d %d", width, height);
      ^

image_to_pgm.cu(82): warning #1650-D: result of call is not used
      fscanf(file, "%d", &maxval);
      ^

image_to_pgm.cu(59): warning #1650-D: result of call is not used
      fscanf(file, "%2s", magic);
      ^

Remark: The warnings can be suppressed with "-diag-suppress <warning-number>"

image_to_pgm.cu(79): warning #1650-D: result of call is not used
      fscanf(file, "%d %d", width, height);
      ^

image_to_pgm.cu(82): warning #1650-D: result of call is not used
      fscanf(file, "%d", &max

# Esecuzione del kernel per conversione da ppm a pgm
### usage:
!./img_to_pgm <input.ppm> <output.ppm>

In [ ]:
!./img_to_pgm NSB.ppm NSB.pgm

=== Conversione Immagine a PGM con CUDA ===
Input: NSB.ppm
Output: NSB.pgm

Dimensioni immagine: 970 x 546
Canali: 3
Grid: (61, 35), Block: (16, 16)

Esecuzione kernel CUDA...
Kernel completato!

Tempo rgbtogray: 114.993149 ms
Scrittura file PGM...
Conversione completata con successo!
File salvato: NSB.pgm


# Compilazione soluzione naive


In [6]:
!nvcc -O3 -lineinfo DFTcudanaive.cu -o dftnaive

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
DFTcudanaive.cu(51): warning #1650-D: result of call is not used
      fscanf(file, "%s", format);
      ^

Remark: The warnings can be suppressed with "-diag-suppress <warning-number>"

DFTcudanaive.cu(57): warning #1650-D: result of call is not used
      fscanf(file, "%d %d", &img.width, &img.height);
      ^

DFTcudanaive.cu(58): warning #1650-D: result of call is not used
      fscanf(file, "%d", &img.max_value);
      ^

DFTcudanaive.cu(62): warning #1650-D: result of call is not used
      fread(img.data, 1, img.width * img.height, file);
      ^

DFTcudanaive.cu(51): warning #1650-D: result of call is not used
      fscanf(file, "%s", format);
      ^

Remark: The warnings can be suppressed with "-diag-suppress <warning-number>"

DFTcudanaive.cu(57): warning #1650-D: result of call is not used


# Esecuzione della soluzione naive 
### Baseline del progetto, realizzata concentrandosi sul solo funzionamento della soluzione e non sull'efficienza
"-f" fa sovrascrittura del file nsight se gia presente su drive

In [8]:
!ncu --set full -f -o test_report ./dftnaive Bologna-512.pgm

Immagine caricata di dimensioni H:340 W:512
==PROF== Connected to process 14096 (/content/drive/MyDrive/ColabNotebooks/dftnaive)
==PROF== Profiling "dft2D" - 0: 0%
==WARNING== Launching the workload is taking more time than expected. If this continues to hang, terminate the profile and re-try by profiling the range of all related launches using '--replay-mode range'. See https://docs.nvidia.com/nsight-compute/ProfilingGuide/index.html#replay for more details.
==PROF== Trying to shutdown target application

==ERROR== Failed to profile "dft2D" in process 14096
==PROF== Trying to shutdown target application
==ERROR== An error occurred while trying to profile.


# Prima ottimizzazione: 
### sono stati usati stratagemmi per ottimizzare l'efficienza dei calcoli matematici
- sincosf
- riduzione della precisione da double a float
- precalcolo delle componenti sin/cos per verticale (costante su u)
- uso di "-use_fast_math" come argomento del compilatore per riconoscere operazioni adatte al fused multiply add(FMA) e sostituire "sincosf" con un'operazione di basso livello hardware "__sincosf", piu veloce a discapito di una leggera perdita di precisione 


In [ ]:
!nvcc -O2 -use_fast_math -lineinfo DFTcudaOpt1.cu -o dftopt1

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
DFTcudaOpt1.cu(51): warning #1650-D: result of call is not used
      fscanf(file, "%s", format);
      ^

Remark: The warnings can be suppressed with "-diag-suppress <warning-number>"

DFTcudaOpt1.cu(57): warning #1650-D: result of call is not used
      fscanf(file, "%d %d", &img.width, &img.height);
      ^

DFTcudaOpt1.cu(58): warning #1650-D: result of call is not used
      fscanf(file, "%d", &img.max_value);
      ^

DFTcudaOpt1.cu(62): warning #1650-D: result of call is not used
      fread(img.data, 1, img.width * img.height, file);
      ^

DFTcudaOpt1.cu(51): warning #1650-D: result of call is not used
      fscanf(file, "%s", format);
      ^

Remark: The warnings can be suppressed with "-diag-suppress <warning-number>"

DFTcudaOpt1.cu(57): warning #1650-D: result of call is not used
      

esecuzione test

In [10]:
!./dftopt1 Bologna-512.pgm

/bin/bash: line 1: ./dftopt1: Permission denied


### Esecuzione con salvataggio dati per nsight 

In [7]:
!ncu --set full -f -o test_report_opt1 ./dftopt1 Bologna-512.pgm

Immagine caricata di dimensioni H:340 W:512
==PROF== Connected to process 8582 (/content/drive/MyDrive/ColabNotebooks/dftopt1)
==PROF== Profiling "dft2D_opt" - 0: 0%.
==WARNING== Launching the workload is taking more time than expected. If this continues to hang, terminate the profile and re-try by profiling the range of all related launches using '--replay-mode range'. See https://docs.nvidia.com/nsight-compute/ProfilingGuide/index.html#replay for more details.
...50%....100% - 31 passes
==PROF== Profiling "filtro" - 1: 0%....50%....100% - 31 passes
==PROF== Profiling "idft2D_opt" - 2: 0%.
==WARNING== Launching the workload is taking more time than expected. If this continues to hang, terminate the profile and re-try by profiling the range of all related launches using '--replay-mode range'. See https://docs.nvidia.com/nsight-compute/ProfilingGuide/index.html#replay for more details.
...50%....100% - 31 passes
Trasformata e antitrasformata completate. Risultato salvato in output_cuda-

# Seconda ottimizzazione:
### Uso della shared memory
Viene fornitaa shared memory a griglie 16x16 di thread

In [9]:
!nvcc -O2 -use_fast_math -lineinfo DFTcudaOpt2.cu -o dftopt2

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
DFTcudaOpt2.cu:84: warning: "PI" redefined
   84 | #define PI 3.14159265358979323846f
      | 
DFTcudaOpt2.cu:7: note: this is the location of the previous definition
    7 | #define PI 3.14159265358979323846
      | 
DFTcudaOpt2.cu(52): warning #1650-D: result of call is not used
      fscanf(file, "%s", format);
      ^

Remark: The warnings can be suppressed with "-diag-suppress <warning-number>"

DFTcudaOpt2.cu(58): warning #1650-D: result of call is not used
      fscanf(file, "%d %d", &img.width, &img.height);
      ^

DFTcudaOpt2.cu(59): warning #1650-D: result of call is not used
      fscanf(file, "%d", &img.max_value);
      ^

DFTcudaOpt2.cu(63): warning #1650-D: result of call is not used
      fread(img.data, 1, img.width * img.height, file);
      ^

DFTcudaOpt2.cu:84: warning: "PI" redef

esecuzione test

In [8]:
!./dftopt2 Bologna-512.pgm

Immagine caricata di dimensioni H:340 W:512
Trasformata e antitrasformata completate. Risultato salvato in output_cuda-512.pgm


### Esecuzione con salvataggio dati per nsight

In [10]:
!ncu --set full -f -o test_report_opt2 ./dftopt2 Bologna-512.pgm

Immagine caricata di dimensioni H:340 W:512
==PROF== Connected to process 15189 (/content/drive/MyDrive/ColabNotebooks/dftopt2)
==PROF== Profiling "dft2D_opt_shared" - 0: 0%.
==WARNING== Launching the workload is taking more time than expected. If this continues to hang, terminate the profile and re-try by profiling the range of all related launches using '--replay-mode range'. See https://docs.nvidia.com/nsight-compute/ProfilingGuide/index.html#replay for more details.
...50%....100% - 31 passes
==PROF== Profiling "filtro" - 1: 0%....50%....100% - 31 passes
==PROF== Profiling "idft2D_opt_shared" - 2: 0%.
==WARNING== Launching the workload is taking more time than expected. If this continues to hang, terminate the profile and re-try by profiling the range of all related launches using '--replay-mode range'. See https://docs.nvidia.com/nsight-compute/ProfilingGuide/index.html#replay for more details.
...50%....100% - 31 passes
Trasformata e antitrasformata completate. Risultato salvato 

# Terza ottimizzazione
## Vari test con loop unrolling
l'unrolling viene fatto specificando al compilatore tramite keyword #pragma unroll 'n', dove n è il numero di interazioni in cui spezzare il loop, nel nostro caso operando sul loop interno di dft e idft in seguito alla definizione dei tile come 16x16, faremo unroll per divisore di 16, ovvero 4, 8 e 16


# Test 1: unroll 4



In [17]:
!nvcc -O2 -use_fast_math -lineinfo DFTcudaOpt3.cu -o dftopt3

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
DFTcudaOpt3.cu:84: warning: "PI" redefined
   84 | #define PI 3.14159265358979323846f
      | 
DFTcudaOpt3.cu:7: note: this is the location of the previous definition
    7 | #define PI 3.14159265358979323846
      | 
DFTcudaOpt3.cu(52): warning #1650-D: result of call is not used
      fscanf(file, "%s", format);
      ^

Remark: The warnings can be suppressed with "-diag-suppress <warning-number>"

DFTcudaOpt3.cu(58): warning #1650-D: result of call is not used
      fscanf(file, "%d %d", &img.width, &img.height);
      ^

DFTcudaOpt3.cu(59): warning #1650-D: result of call is not used
      fscanf(file, "%d", &img.max_value);
      ^

DFTcudaOpt3.cu(63): warning #1650-D: result of call is not used
      fread(img.data, 1, img.width * img.height, file);
      ^

DFTcudaOpt3.cu:84: warning: "PI" redef

test:

In [18]:
!./dftopt3 Bologna-512.pgm

Immagine caricata di dimensioni H:340 W:512
Trasformata e antitrasformata completate. Risultato salvato in output_cuda-512.pgm


esecuzione con raccolta report

In [19]:
!ncu --set full -f -o test_report_opt3 ./dftopt3 Bologna-512.pgm

Immagine caricata di dimensioni H:340 W:512
==PROF== Connected to process 18218 (/content/drive/MyDrive/ColabNotebooks/dftopt3)
==PROF== Profiling "dft2D_opt_shared" - 0: 0%.
==WARNING== Launching the workload is taking more time than expected. If this continues to hang, terminate the profile and re-try by profiling the range of all related launches using '--replay-mode range'. See https://docs.nvidia.com/nsight-compute/ProfilingGuide/index.html#replay for more details.
...50%....100% - 31 passes
==PROF== Profiling "filtro" - 1: 0%....50%....100% - 31 passes
==PROF== Profiling "idft2D_opt_shared" - 2: 0%.
==WARNING== Launching the workload is taking more time than expected. If this continues to hang, terminate the profile and re-try by profiling the range of all related launches using '--replay-mode range'. See https://docs.nvidia.com/nsight-compute/ProfilingGuide/index.html#replay for more details.
...50%....100% - 31 passes
Trasformata e antitrasformata completate. Risultato salvato 

# Test 2: unroll 8



In [20]:
!nvcc -O2 -use_fast_math -lineinfo DFTcudaOpt4.cu -o dftopt4

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
DFTcudaOpt4.cu:84: warning: "PI" redefined
   84 | #define PI 3.14159265358979323846f
      | 
DFTcudaOpt4.cu:7: note: this is the location of the previous definition
    7 | #define PI 3.14159265358979323846
      | 
DFTcudaOpt4.cu(52): warning #1650-D: result of call is not used
      fscanf(file, "%s", format);
      ^

Remark: The warnings can be suppressed with "-diag-suppress <warning-number>"

DFTcudaOpt4.cu(58): warning #1650-D: result of call is not used
      fscanf(file, "%d %d", &img.width, &img.height);
      ^

DFTcudaOpt4.cu(59): warning #1650-D: result of call is not used
      fscanf(file, "%d", &img.max_value);
      ^

DFTcudaOpt4.cu(63): warning #1650-D: result of call is not used
      fread(img.data, 1, img.width * img.height, file);
      ^

DFTcudaOpt4.cu:84: warning: "PI" redef

test:

In [ ]:
!./dftopt4 Bologna-512.pgm

esecuzione con raccolta report

In [21]:
!ncu --set full -f -o test_report_opt4 ./dftopt4 Bologna-512.pgm

Immagine caricata di dimensioni H:340 W:512
==PROF== Connected to process 18713 (/content/drive/MyDrive/ColabNotebooks/dftopt4)
==PROF== Profiling "dft2D_opt_shared" - 0: 0%.
==WARNING== Launching the workload is taking more time than expected. If this continues to hang, terminate the profile and re-try by profiling the range of all related launches using '--replay-mode range'. See https://docs.nvidia.com/nsight-compute/ProfilingGuide/index.html#replay for more details.
...50%....100% - 31 passes
==PROF== Profiling "filtro" - 1: 0%....50%....100% - 31 passes
==PROF== Profiling "idft2D_opt_shared" - 2: 0%....50%....100% - 31 passes
Trasformata e antitrasformata completate. Risultato salvato in output_cuda-512.pgm
==PROF== Disconnected from process 18713
==PROF== Report: /content/drive/MyDrive/ColabNotebooks/test_report_opt4.ncu-rep


# Test 3: unroll 16



In [22]:
!nvcc -O2 -use_fast_math -lineinfo DFTcudaOpt5.cu -o dftopt5

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
DFTcudaOpt5.cu:84: warning: "PI" redefined
   84 | #define PI 3.14159265358979323846f
      | 
DFTcudaOpt5.cu:7: note: this is the location of the previous definition
    7 | #define PI 3.14159265358979323846
      | 
DFTcudaOpt5.cu(52): warning #1650-D: result of call is not used
      fscanf(file, "%s", format);
      ^

Remark: The warnings can be suppressed with "-diag-suppress <warning-number>"

DFTcudaOpt5.cu(58): warning #1650-D: result of call is not used
      fscanf(file, "%d %d", &img.width, &img.height);
      ^

DFTcudaOpt5.cu(59): warning #1650-D: result of call is not used
      fscanf(file, "%d", &img.max_value);
      ^

DFTcudaOpt5.cu(63): warning #1650-D: result of call is not used
      fread(img.data, 1, img.width * img.height, file);
      ^

DFTcudaOpt5.cu:84: warning: "PI" redef

test:

In [ ]:
!./dftopt5 Bologna-512.pgm

esecuzione con raccolta report

In [23]:
!ncu --set full -f -o test_report_opt5 ./dftopt5 Bologna-512.pgm

Immagine caricata di dimensioni H:340 W:512
==PROF== Connected to process 19194 (/content/drive/MyDrive/ColabNotebooks/dftopt5)
==PROF== Profiling "dft2D_opt_shared" - 0: 0%.
==WARNING== Launching the workload is taking more time than expected. If this continues to hang, terminate the profile and re-try by profiling the range of all related launches using '--replay-mode range'. See https://docs.nvidia.com/nsight-compute/ProfilingGuide/index.html#replay for more details.
...50%....100% - 31 passes
==PROF== Profiling "filtro" - 1: 0%....50%....100% - 31 passes
==PROF== Profiling "idft2D_opt_shared" - 2: 0%.
==WARNING== Launching the workload is taking more time than expected. If this continues to hang, terminate the profile and re-try by profiling the range of all related launches using '--replay-mode range'. See https://docs.nvidia.com/nsight-compute/ProfilingGuide/index.html#replay for more details.
...50%....100% - 31 passes
Trasformata e antitrasformata completate. Risultato salvato 

### Si è notato un incremento delle performance solo nel momento in cui è stato fatto loop unrolling di 16 istruzione, dato che va a arimuovere completamente l'operazione condizionale del ciclo

# Quarta ottimizzazione
### Cache L1 configuration